# Clase 211 — Polars: lazy API, streaming, benchmarks

Requiere: `pip install polars pyarrow pandas duckdb`.

In [ ]:
import polars as pl, pandas as pd, time, tempfile, os
from pathlib import Path

WORK = Path(tempfile.gettempdir()) / 'polars_demo'
WORK.mkdir(exist_ok=True)
print('Polars version:', pl.__version__)

## 1. Dataset sintético — 5M filas

In [ ]:
import numpy as np
rng = np.random.default_rng(42)
N = 5_000_000
df_pl = pl.DataFrame({
    'zone_id': rng.integers(0, 100, N),
    'fare':    rng.uniform(5, 100, N),
    'tip':     rng.uniform(0, 20, N),
    'date':    pl.date(2024, 1, 1) + pl.duration(days=pl.Series(rng.integers(0, 60, N))),
    'borough': rng.choice(['Manhattan', 'Brooklyn', 'Queens', 'Bronx'], N),
})

pq = WORK / 'trips.parquet'
df_pl.write_parquet(pq)
print(f'parquet: {pq.stat().st_size / 1024 / 1024:.1f} MB, {N:,} filas')

## 2. Pandas vs Polars eager vs Polars lazy

In [ ]:
# Misma query: agregado por borough con filtro
def bench(name, fn):
    t0 = time.perf_counter()
    out = fn()
    dt = time.perf_counter() - t0
    print(f'{name:25} {dt*1000:>8.1f} ms')
    return out

bench('pandas', lambda: (pd.read_parquet(pq).query('fare > 30').groupby('borough')
                          .agg(avg_fare=('fare', 'mean'), n=('fare', 'size'))))

bench('polars eager', lambda: (pl.read_parquet(pq).filter(pl.col('fare') > 30)
                                .group_by('borough').agg(pl.col('fare').mean().alias('avg_fare'),
                                                          pl.len().alias('n'))))

bench('polars lazy', lambda: (pl.scan_parquet(pq).filter(pl.col('fare') > 30)
                               .group_by('borough').agg(pl.col('fare').mean().alias('avg_fare'),
                                                         pl.len().alias('n'))
                               .collect()))

## 3. Optimizaciones del query planner

In [ ]:
q = (pl.scan_parquet(pq)
     .filter(pl.col('borough') == 'Manhattan')
     .filter(pl.col('fare').is_between(10, 50))
     .select('zone_id', 'fare', 'tip')
     .group_by('zone_id').agg(pl.col('fare').mean(), pl.col('tip').mean()))

print('=== Plan optimizado ===')
print(q.explain())
print('\n→ Observá: PROJECT solo 4 columnas (column pruning),')
print('  filter pushed down al PARQUET SCAN (predicate pushdown).')

## 4. Streaming engine (datasets > RAM)

In [ ]:
# Streaming es importante con datasets que NO caben en RAM.
# Acá funciona igual; en datasets de 100+ GB es la diferencia entre OK y OOM.
t0 = time.perf_counter()
out = q.collect(engine='streaming')
print(f'streaming engine: {(time.perf_counter() - t0) * 1000:.1f} ms')
print(out.head())

## 5. Window functions con `over()`

In [ ]:
# Rolling mean de fare por borough, ordenado por date
result = (df_pl.sort('date')
          .with_columns([
              pl.col('fare').rolling_mean(window_size=10000).over('borough').alias('rolling_avg_fare'),
              (pl.col('fare') - pl.col('fare').mean().over('borough')).alias('fare_dev_from_borough_mean'),
          ]))
result.select('date', 'borough', 'fare', 'rolling_avg_fare', 'fare_dev_from_borough_mean').head()

## 6. Interop con DuckDB (zero-copy via Arrow)

In [ ]:
import duckdb
con = duckdb.connect()

# DuckDB lee Polars directo (Arrow zero-copy)
sql_result = con.execute('''
    SELECT borough, AVG(fare) AS avg_fare, COUNT(*) AS n
    FROM df_pl
    WHERE fare > 30
    GROUP BY borough
    ORDER BY avg_fare DESC
''').pl()   # devolver como Polars DataFrame
print(sql_result)
con.close()

## Ejercicio guiado

1. Migrá un script pandas tuyo a Polars. Medí speedup. Casos donde NO sea más rápido: documentar por qué.
2. Usá `.explain()` antes y después de cambiar el orden de filters/selects — observá que el optimizer da el mismo plan.
3. Generá un parquet de 5 GB (loop generando chunks) y procesalo con `engine='streaming'`. Medí RAM peak con `psutil`.
4. Combiná Polars + DuckDB: feature engineering en Polars, query analítica final en SQL DuckDB sobre el resultado.
5. Bonus: integrá Polars en un flow Prefect (Clase 209).

## Conclusiones

- Polars eager ≈ 3-10× pandas; Polars lazy ≈ 5-30× pandas en pipelines reales.
- Lazy es para producción; eager para REPL.
- Streaming engine permite out-of-core sin saltar a Spark.
- Arrow zero-copy hace que el interop con DuckDB / pandas / PyArrow sea gratis.